In [1]:
import pandas as pd

In [2]:
df_raw = pd.read_csv("/content/eng_sentences.tsv", sep='\t')
df_raw.head()

,1276,eng,Let's try something.
0,1277,eng,I have to go to sleep.
1,1280,eng,Today is June 18th and it is Muiriel's birthday!
2,1282,eng,Muiriel is 20 now.
3,1283,eng,"The password is ""Muiriel""."
4,1284,eng,I will be back soon.


In [3]:
df_raw.shape

(1951597, 3)

In [4]:
# This code is generated by Claude to be honest
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r"[^a-z\s']", '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_raw["Let's try something."] = df_raw["Let's try something."].astype(str).apply(clean_text)
df_raw.head()

,1276,eng,Let's try something.
0,1277,eng,i have to go to sleep
1,1280,eng,today is june th and it is muiriel's birthday
2,1282,eng,muiriel is now
3,1283,eng,the password is muiriel
4,1284,eng,i will be back soon


In [5]:
# calculate our vocab size
sentences = []
for i in range(194379):
    sentences.append(df_raw["Let's try something."][i])

vocab = set()
for sentence in sentences:
    for word in sentence.split():
        vocab.add(word)

len(vocab)

31084

In [6]:
# Now we need to check number of occurences of each word
# This code is generated by Claude
from collections import Counter

all_words = [word for sentence in sentences for word in sentence.split()]
occ = Counter(all_words)

In [7]:
occ

Counter({'i': 37802,
         'have': 11268,
         'to': 51367,
         'go': 4192,
         'sleep': 420,
         'today': 1291,
         'is': 33995,
         'june': 92,
         'th': 165,
         'and': 12760,
         'it': 15132,
         "muiriel's": 4,
         'birthday': 242,
         'muiriel': 15,
         'now': 2132,
         'the': 85614,
         'password': 12,
         'will': 6860,
         'be': 8381,
         'back': 1513,
         'soon': 1083,
         "i'm": 3438,
         'at': 9411,
         'a': 41678,
         'loss': 159,
         'for': 14737,
         'words': 633,
         'this': 11583,
         'never': 1789,
         'going': 2137,
         'end': 581,
         'just': 2031,
         "don't": 4882,
         'know': 2915,
         'what': 5891,
         'say': 1527,
         'that': 14544,
         'was': 15677,
         'an': 4769,
         'evil': 100,
         'bunny': 2,
         'in': 25634,
         'mountains': 117,
         'recent': 73,

In [8]:
# Generated by Claude
sorted_rare = sorted(
    ((w, c) for w, c in occ.items() if c <= 2),
    key=lambda x: x[1]
)

In [9]:
# Generated by Claude
rare_words = {w for w, c in sorted_rare}
vocab = vocab - rare_words

len(vocab)

14086

In [10]:
# This code generally generated by Andrej Karpathy
special_tokens = ["<pad>", "<unk>"]
vocab_full = special_tokens + [w for w in vocab if w not in special_tokens]

stoi = {word: i for i, word in enumerate(vocab_full)}
itos = {i: word for i, word in enumerate(vocab_full)}

In [11]:
stoi

{'<pad>': 0,
 '<unk>': 1,
 'astonishment': 2,
 'bathed': 3,
 'fourletter': 4,
 'boring': 5,
 'pleaded': 6,
 'granddaughter': 7,
 'decreasing': 8,
 'unturned': 9,
 'stronger': 10,
 'spider': 11,
 'crammed': 12,
 "kelly's": 13,
 'retorted': 14,
 'hitherto': 15,
 'politically': 16,
 'tuning': 17,
 'freed': 18,
 'runny': 19,
 'okinawa': 20,
 'expecting': 21,
 'constantly': 22,
 'aback': 23,
 'james': 24,
 'okay': 25,
 'jig': 26,
 'tutor': 27,
 'lighted': 28,
 'vietnamese': 29,
 'planned': 30,
 "what's": 31,
 'empire': 32,
 'newspaperman': 33,
 'macedonian': 34,
 'vocal': 35,
 'village': 36,
 'aftermath': 37,
 "someone's": 38,
 'stale': 39,
 'charged': 40,
 'count': 41,
 'links': 42,
 'bee': 43,
 'outside': 44,
 'director': 45,
 'inevitably': 46,
 'free': 47,
 'helmet': 48,
 'symbols': 49,
 'absolved': 50,
 'cheering': 51,
 'nepal': 52,
 'zuckerberg': 53,
 'raise': 54,
 'surgery': 55,
 'twohour': 56,
 'replacing': 57,
 'ridden': 58,
 'strap': 59,
 'premises': 60,
 'extension': 61,
 'appoint

In [12]:
# Claude improved my function btw. I 100% knows how does it work!
def create_dataset(sentence, stoi):
    X, Y = [], []
    sentence = ["<pad>", "<pad>"] + sentence.split() + ["<pad>", "<pad>"]
    UNK = stoi["<unk>"]
    for i in range(2, len(sentence) - 2):
        context = [stoi.get(sentence[j], UNK) for j in (i-2, i-1, i+1, i+2)]
        target = stoi.get(sentence[i], UNK)
        X.append(context)
        Y.append(target)
    return X, Y

X, Y = [], []
for s in sentences:
    xs, ys = create_dataset(s, stoi)
    X.extend(xs)
    Y.extend(ys)

In [13]:
X

[[0, 0, 9343, 12587],
 [0, 8612, 12587, 3562],
 [8612, 9343, 3562, 12587],
 [9343, 12587, 12587, 7254],
 [12587, 3562, 7254, 0],
 [3562, 12587, 0, 0],
 [0, 0, 13763, 1569],
 [0, 8255, 1569, 1723],
 [8255, 13763, 1723, 11937],
 [13763, 1569, 11937, 4667],
 [1569, 1723, 4667, 13763],
 [1723, 11937, 13763, 2278],
 [11937, 4667, 2278, 5820],
 [4667, 13763, 5820, 0],
 [13763, 2278, 0, 0],
 [0, 0, 13763, 6416],
 [0, 6415, 6416, 0],
 [6415, 13763, 0, 0],
 [0, 0, 4542, 13763],
 [0, 12503, 13763, 6415],
 [12503, 4542, 6415, 0],
 [4542, 13763, 0, 0],
 [0, 0, 10239, 5461],
 [0, 8612, 5461, 8916],
 [8612, 10239, 8916, 912],
 [10239, 5461, 912, 0],
 [5461, 8916, 0, 0],
 [0, 0, 81, 8173],
 [0, 5160, 8173, 365],
 [5160, 81, 365, 10397],
 [81, 8173, 10397, 145],
 [8173, 365, 145, 0],
 [365, 10397, 0, 0],
 [0, 0, 13763, 12718],
 [0, 4787, 12718, 1003],
 [4787, 13763, 1003, 12587],
 [13763, 12718, 12587, 10036],
 [12718, 1003, 10036, 0],
 [1003, 12587, 0, 0],
 [0, 0, 310, 9993],
 [0, 8612, 9993, 8536],


In [14]:
Y

[8612,
 9343,
 12587,
 3562,
 12587,
 7254,
 8255,
 13763,
 1569,
 1723,
 11937,
 4667,
 13763,
 2278,
 5820,
 6415,
 13763,
 6416,
 12503,
 4542,
 13763,
 6415,
 8612,
 10239,
 5461,
 8916,
 912,
 5160,
 81,
 8173,
 365,
 10397,
 145,
 4787,
 13763,
 12718,
 1003,
 12587,
 10036,
 8612,
 310,
 9993,
 8536,
 12963,
 12587,
 11435,
 13634,
 13327,
 8951,
 9074,
 1,
 8612,
 13327,
 8748,
 12503,
 6411,
 13763,
 4667,
 8173,
 1609,
 2662,
 8612,
 9993,
 8536,
 1693,
 8612,
 9343,
 12503,
 12748,
 9435,
 8748,
 4787,
 2274,
 1,
 13120,
 6531,
 8748,
 2444,
 2546,
 10255,
 8612,
 2096,
 3677,
 978,
 8748,
 4236,
 14042,
 4787,
 10239,
 6617,
 8612,
 5712,
 8173,
 290,
 8612,
 1902,
 4533,
 12793,
 912,
 11937,
 310,
 11137,
 2767,
 12294,
 4612,
 3677,
 9993,
 13957,
 12587,
 5461,
 4148,
 13634,
 12212,
 2178,
 8649,
 1894,
 12035,
 5461,
 8173,
 281,
 3831,
 7636,
 6187,
 4236,
 12671,
 6513,
 12587,
 2214,
 12318,
 592,
 8612,
 12035,
 1896,
 5448,
 1693,
 4787,
 13763,
 12503,
 3836,
 1

In [15]:
# Generated by Claude

import torch
from torch.utils.data import Dataset, DataLoader

class CBOWDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.Y = torch.tensor(Y, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

dataset = CBOWDataset(X, Y)
loader = DataLoader(dataset, batch_size=256, shuffle=True)

In [16]:
# Generated by Claude

import torch.nn as nn
class CBOW(nn.Module):
    def __init__(self, vocab_size, embed_dim, pad_idx):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.linear = nn.Linear(embed_dim, vocab_size)

    def forward(self, context):
        embeds = self.embeddings(context)
        avg = embeds.mean(dim=1)
        logits = self.linear(avg)
        return logits

In [31]:
# Walkthrough 1
embed = nn.Embedding(len(vocab_full), 4, padding_idx=stoi["<pad>"])
linear = nn.Linear(4, len(vocab_full))
embeds = embed(torch.tensor(X[0]))
embeds

tensor([[ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.3278,  0.8168, -0.5890, -0.6669],
        [-0.7513,  1.1689, -0.6213,  2.9871]], grad_fn=<EmbeddingBackward0>)

In [32]:
# Walkthrough 2
avg = embeds.mean(dim=1)
avg

tensor([ 0.0000,  0.0000, -0.0278,  0.6958], grad_fn=<MeanBackward1>)

In [33]:
# Walkthrouhg 3
logits = linear(avg)
logits

tensor([ 0.3195, -0.2494, -0.2449,  ...,  0.1322, -0.1461, -0.1140],
       grad_fn=<ViewBackward0>)

In [34]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VOCAB_SIZE = len(stoi)
EMBED_DIM = 100
PAD_IDX = stoi["<pad>"]
BATCH_SIZE = 256
EPOCHS = 10
LR = 0.01

In [35]:
model = CBOW(VOCAB_SIZE, EMBED_DIM, PAD_IDX).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [36]:
for epoch in range(EPOCHS):
    total_loss = 0.0
    for context, target in loader:
        context, target = context.to(device), target.to(device)

        optimizer.zero_grad()
        logits = model(context)
        loss = criterion(logits, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {avg_loss:.4f}")

Epoch 1/10, Loss: 5.1704
Epoch 2/10, Loss: 4.4842
Epoch 3/10, Loss: 4.2862
Epoch 4/10, Loss: 4.1902
Epoch 5/10, Loss: 4.1360
Epoch 6/10, Loss: 4.1011
Epoch 7/10, Loss: 4.0803
Epoch 8/10, Loss: 4.0632
Epoch 9/10, Loss: 4.0502
Epoch 10/10, Loss: 4.0414


In [44]:
logits = model(torch.tensor(X[1]).unsqueeze(0).to(device))
predicted_id = torch.argmax(logits, dim=1)
itos[predicted_id.item()]

'want'

In [42]:
for i in X[1]:
    print(itos[i])

<pad>
i
to
go


In [43]:
itos[Y[1]]

'have'